## Model A — ResNet-34 Inference / Submission

Inference notebook for the ResNet-34 model. Loads the ONNX model from dataset, runs CPU inference on test soundscapes, outputs `submission.csv`.

In [1]:
# ── DEBUG: verify all mounted paths ──────────────────────────────────────
import os
print("=== /kaggle/input contents ===")
for d in sorted(os.listdir('/kaggle/input')):
    sub = os.listdir(f'/kaggle/input/{d}')
    print(f"  {d}/  →  {sub[:6]}{'...' if len(sub)>6 else ''}")

=== /kaggle/input contents ===
  competitions/  →  ['birdclef-2026']
  datasets/  →  ['yaneshumliu']
  notebooks/  →  ['ttahara']


In [2]:
# ═══════════════════════════════════════════════════════════════════════════
#  STEP 1 — Imports
# ═══════════════════════════════════════════════════════════════════════════
import os, json, glob, warnings
from pathlib import Path

import numpy as np
import pandas as pd
from tqdm.auto import tqdm

import torch
import torch.nn as nn
import torch.nn.functional as F
import torchaudio
import torchaudio.transforms as T
import timm

warnings.filterwarnings("ignore")
print(f"torch      : {torch.__version__}")
print(f"torchaudio : {torchaudio.__version__}")
print(f"timm       : {timm.__version__}")

# ═══════════════════════════════════════════════════════════════════════════
#  STEP 2 — Config
# ═══════════════════════════════════════════════════════════════════════════
_BASE_DIR = None
for _candidate in ["/kaggle/input/birdclef-2026",
                   "/kaggle/input/competitions/birdclef-2026"]:
    if Path(_candidate).exists():
        _BASE_DIR = _candidate
        break

if _BASE_DIR is None:
    raise RuntimeError("Competition data not found.")

class CFG:
    TEST_DIR    = f"{_BASE_DIR}/test_soundscapes"
    SAMPLE_SUB  = f"{_BASE_DIR}/sample_submission.csv"

    DATASET_DIR  = "/kaggle/input/datasets/yaneshumliu/12-epoch-data"
    
    # Використовуємо .pth замість .onnx
    WEIGHTS_PATH = f"{DATASET_DIR}/bird_sed_resnet34.pth" 
    META_PATH    = f"{DATASET_DIR}/target_columns.json"

    MODEL_NAME  = "resnet34"
    TARGET_SR   = 32_000
    SEGMENT_SEC = 5.0
    N_MELS      = 128
    N_FFT       = 1024
    HOP_LENGTH  = 320
    FMIN        = 20.0
    FMAX        = 16_000.0
    
    DEVICE      = torch.device("cuda" if torch.cuda.is_available() else "cpu")

cfg = CFG()

with open(cfg.META_PATH) as f:
    INF_TARGET_COLUMNS = json.load(f)
NUM_CLASSES = len(INF_TARGET_COLUMNS)
print(f"Loaded {NUM_CLASSES} bird classes.")

if not Path(cfg.WEIGHTS_PATH).exists():
    raise FileNotFoundError(f"PyTorch weights not found at {cfg.WEIGHTS_PATH}.")

# ═══════════════════════════════════════════════════════════════════════════
#  STEP 3 — Model Architecture (From your Notebook)
# ═══════════════════════════════════════════════════════════════════════════
class BirdSEDModel(nn.Module):
    def __init__(self, model_name: str, num_classes: int, pretrained: bool = False):
        super().__init__()
        self.backbone = timm.create_model(
            model_name,
            pretrained=pretrained, # False для сабміту (без інтернету)
            in_chans=1,
            num_classes=0,
            global_pool="",
        )

        # Зондування розмірності ознак
        with torch.no_grad():
            dummy = torch.zeros(1, 1, 128, 312)
            feat  = self.backbone(dummy)
            in_ch = feat.shape[1] 

        self.dropout = nn.Dropout(0.3)
        self.fc_clip = nn.Linear(in_ch, num_classes)   
        self.fc_att  = nn.Linear(in_ch, num_classes)   

    def forward(self, x: torch.Tensor) -> torch.Tensor:
        feat = self.backbone(x)            
        feat = feat.mean(dim=2)             
        feat = feat.permute(0, 2, 1)        
        feat = self.dropout(feat)

        clip_logits = self.fc_clip(feat)                         
        att_weights = torch.softmax(self.fc_att(feat), dim=1)    

        # Модель вже робить sigmoid!
        out = (torch.sigmoid(clip_logits) * att_weights).sum(dim=1)
        return out

# ═══════════════════════════════════════════════════════════════════════════
#  STEP 4 — Helper functions (Native PyTorch)
# ═══════════════════════════════════════════════════════════════════════════
def build_mel_transforms(cfg):
    mel = T.MelSpectrogram(
        sample_rate=cfg.TARGET_SR, n_fft=cfg.N_FFT, hop_length=cfg.HOP_LENGTH,
        n_mels=cfg.N_MELS, f_min=cfg.FMIN, f_max=cfg.FMAX,
    )
    db = T.AmplitudeToDB(stype="power", top_db=80)
    return mel, db

def audio_to_spec(wav_segment, mel_t, db_t):
    spec = db_t(mel_t(wav_segment))
    spec = (spec - spec.mean()) / (spec.std() + 1e-6)
    return spec.unsqueeze(0)  # Tensor shape: (1, 1, N_MELS, T)

def predict_file(path, model, mel_t, db_t, cfg):
    fname = Path(path).stem
    try:
        wav, sr = torchaudio.load(path)
    except Exception as e:
        print(f"[ERROR] Cannot load {path}: {e}")
        return []

    if wav.shape[0] > 1:
        wav = wav.mean(dim=0, keepdim=True)
    if sr != cfg.TARGET_SR:
        wav = T.Resample(orig_freq=sr, new_freq=cfg.TARGET_SR)(wav)

    seg_len    = int(cfg.SEGMENT_SEC * cfg.TARGET_SR)
    n_windows  = max(1, int(np.ceil(wav.shape[1] / seg_len)))
    results    = []

    for i in range(n_windows):
        start   = i * seg_len
        segment = wav[:, start : start + seg_len]
        if segment.shape[1] < seg_len:
            segment = F.pad(segment, (0, seg_len - segment.shape[1]))

        spec = audio_to_spec(segment, mel_t, db_t).to(cfg.DEVICE)
        
        with torch.no_grad():
            # Отримуємо ймовірності напряму (sigmoid вже всередині моделі)
            probs = model(spec).squeeze(0).cpu().numpy()
            
        results.append((f"{fname}_{(i + 1) * 5}", probs))

    return results

# ═══════════════════════════════════════════════════════════════════════════
#  STEP 5 — PyTorch Inference & Safe Submission Logic
# ═══════════════════════════════════════════════════════════════════════════
print(f"Loading PyTorch model to {cfg.DEVICE}...")
model = BirdSEDModel(model_name=cfg.MODEL_NAME, num_classes=NUM_CLASSES, pretrained=False)
state_dict = torch.load(cfg.WEIGHTS_PATH, map_location=cfg.DEVICE, weights_only=True)

# Завантажуємо ваші натреновані ваги в "порожній каркас"
model.load_state_dict(state_dict, strict=False)
model.to(cfg.DEVICE)
model.eval()
print("Model loaded successfully.")

mel_t, db_t = build_mel_transforms(cfg)
test_files  = sorted(glob.glob(f"{cfg.TEST_DIR}/*.ogg"))

sample_sub   = pd.read_csv(cfg.SAMPLE_SUB)
SUB_COLS     = list(sample_sub.columns)        
SPECIES_COLS = SUB_COLS[1:]                   

print(f"Test soundscapes found: {len(test_files)}")

if not test_files:
    print("[INFO] Commit mode — saving dummy submission.")
    sample_sub.to_csv("submission.csv", index=False)
else:
    predictions = {}
    for fp in tqdm(test_files, desc="Inference"):
        rows = predict_file(fp, model, mel_t, db_t, cfg)
        for row_id, probs in rows:
            predictions[row_id] = probs

    # Створюємо мапу для правильного співставлення ваших класів та колонок Kaggle
    model_cols_set = set(INF_TARGET_COLUMNS)
    col_idx_map = {col: INF_TARGET_COLUMNS.index(col) for col in SPECIES_COLS if col in model_cols_set}

    records = []
    missing_rows = 0
    # Проходимо СУВОРО по рядках, які вимагає Kaggle (sample_sub)
    for row_id in sample_sub["row_id"]:
        record = {"row_id": row_id}
        
        if row_id in predictions:
            probs = predictions[row_id]
            for col in SPECIES_COLS:
                record[col] = float(probs[col_idx_map[col]]) if col in col_idx_map else 0.0
        else:
            missing_rows += 1
            for col in SPECIES_COLS:
                record[col] = 0.0
        
        records.append(record)

    if missing_rows > 0:
        print(f"[WARN] {missing_rows} expected rows were not predicted → filled with 0.0")

    submission_df = pd.DataFrame(records)[SUB_COLS]
    
    assert list(submission_df.columns) == SUB_COLS, "Column mismatch!"
    assert len(submission_df) == len(sample_sub), f"Row count mismatch!"
    
    submission_df.to_csv("submission.csv", index=False)
    print(f"submission.csv saved — {len(submission_df):,} rows x {len(submission_df.columns)} cols")

torch      : 2.10.0+cpu
torchaudio : 2.10.0+cpu
timm       : 1.0.25
Loaded 234 bird classes.
Loading PyTorch model to cpu...
Model loaded successfully.
Test soundscapes found: 0
[INFO] Commit mode — saving dummy submission.
